# M3 · Grad-CAM y Grad-CAM++ (arranque, sin análisis)

**TFM · HAM10000** · Responsable: **Christian Zhirzhan**. GPU T4 (también funciona con CPU, más lento).

Modelos: los tres de la v3 con alias `v3-representativo` (ResNet50 s42, DenseNet121 s42, EfficientNetB0 s123). Entradas: `predicciones_test.csv` y `falsos_negativos_mel.csv` de `m2-test-v3-<arch>`, en `m2-evaluacion-test-v3`.

Casos: todos los FN de melanoma de ResNet50; 30 FN de DenseNet121 y de EfficientNetB0 estratificados por clase asignada (nv / bkl / otras); 30 TP de melanoma por red. Una imagen por `lesion_id` en las muestras. Semilla fija.

Por caso, una figura (original, Grad-CAM y Grad-CAM++ superpuestos; para un FN, una fila con la clase predicha y otra con `mel`). Al final, `m3_casos.csv` con las columnas de anotación vacías. **La anotación manual no se hace aquí.** Secretos `DAGSHUB_USER` y `DAGSHUB_TOKEN`.

In [ ]:
BRANCH = "exp/m3-gradcam"
AUTOR = "christian"
ARCHS = ["resnet50", "densenet121", "efficientnetb0"]
SALIDA = "/content/m3_salida"

## 1. Sesión

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Sin GPU"
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
OWNER, REPO = "aReinoso007", "tfm-melanoma-cnn-gradcam"
REPO_DIR = f"/content/{REPO}"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/{OWNER}/{REPO}.git {REPO_DIR}
%cd {REPO_DIR}
!git fetch -q origin
!git checkout -q {BRANCH}
!git pull -q origin {BRANCH}
!git log --oneline -1
!pip install -q -r requirements.txt

In [ ]:
import shutil, time, zipfile
from pathlib import Path
from src.tracking import load_config, file_sha256
from src.data import SPLIT_SHA256_V1
cfg = load_config("configs/m2_v3.yaml")          # hereda base.yaml; trae el experimento de la prueba v3
LOCAL = Path(cfg["paths"]["local_data_dir"])
if not (LOCAL / "images").exists():
    t0 = time.time()
    shutil.copy(cfg["paths"]["dataset_zip"], "/content/HAM10000.zip")
    with zipfile.ZipFile("/content/HAM10000.zip") as z:
        z.extractall(LOCAL)
    os.remove("/content/HAM10000.zip")
    print(f"Descomprimido ({time.time()-t0:.0f} s)")
assert len(list((LOCAL / "images").glob("*.jpg"))) == 10015
assert file_sha256("data/splits/split_v1.csv") == SPLIT_SHA256_V1
print("Datos y split v1 verificados.", "Experimento M3:", cfg["mlflow"]["experiments"]["m3"])

In [ ]:
from google.colab import userdata
from src.tracking import setup_mlflow
os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("DAGSHUB_USER")
os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")
print(setup_mlflow(cfg))

## 2. Mapas por red

Para cada red: comprueba que `v3-representativo` apunta al run y a la semilla esperados, carga el modelo, confirma el nombre de la última capa convolucional, comprueba que la p(mel) recalculada coincide con la del CSV, genera las figuras y registra el run `m3-gradcam-<arch>` (figuras, `m3_casos_<arch>.csv`, versión del modelo, prueba de cordura de Adebayo).

Si una red falla, el bucle se detiene con el error. Se puede relanzar solo esa red.

In [ ]:
from src.casos import lanzar_gradcam
resultados = []
for arch in ARCHS:
    print(f"\n{'='*20} {arch} {'='*20}")
    r = lanzar_gradcam(arch, cfg, salida=SALIDA, autor=AUTOR)
    resultados.append(r)
    c = r["casos"]
    print(f"run={r['run_id']} modelo v{r['modelo_version']} ({r['run_id_modelo']}) dif_max_p_mel={r['dif_max_p_mel']:.2e}")
    print(c.groupby(["tipo_caso", "estrato"]).size().to_string())

### Prueba de cordura (Adebayo et al., 2018)

Correlación de rangos entre el mapa del modelo entrenado y el de una copia con la parte superior de la base aleatorizada (fracción 0 = idéntico, 1 = toda la base), clase `mel`, 5 TP por red. Debe caer claramente al aumentar la fracción. Cuántos mapas aleatorizados salieron vacíos se registra como `adebayo_vacios_*`.

In [ ]:
import pandas as pd
filas = []
for r in resultados:
    for (metodo, f), rho in r["cordura"]["rho"].items():
        filas.append({"red": r["arch"], "método": metodo, "fracción": f, "rho": round(rho, 3),
                      "vacíos": r["cordura"]["vacios"][(metodo, f)]})
tabla = pd.DataFrame(filas)
display(tabla.pivot_table(index=["red", "método"], columns="fracción", values="rho", dropna=False))
display(tabla.pivot_table(index=["red", "método"], columns="fracción", values="vacíos"))


## 3. `m3_casos.csv`

In [ ]:
from src.casos import publicar_casos
casos = pd.concat([r["casos"] for r in resultados], ignore_index=True)
Path(SALIDA).mkdir(parents=True, exist_ok=True)
run_casos = publicar_casos(casos, resultados, cfg, SALIDA, autor=AUTOR)
DRIVE_M3 = Path(cfg["paths"]["drive_root"]) / "m3"
DRIVE_M3.mkdir(parents=True, exist_ok=True)
shutil.copy(f"{SALIDA}/m3_casos.csv", DRIVE_M3 / "m3_casos.csv")
shutil.copytree(f"{SALIDA}/gradcam", DRIVE_M3 / "gradcam", dirs_exist_ok=True)
print("Run m3-gradcam-casos:", run_casos)
print("Copia en Drive:", DRIVE_M3)
print(casos.groupby(["red", "tipo_caso"]).size().unstack().to_string())
casos.head()

In [ ]:
# Comprobaciones finales: sin lesiones repetidas en las muestras, anotación vacía, todas las figuras escritas.
assert (casos[["zona_atencion", "criterios_abcde", "comentario"]] == "").all().all()
assert all(Path(SALIDA, f).exists() for f in casos.figura)
for (red, tipo), g in casos.groupby(["red", "tipo_caso"]):
    if not (red == "resnet50" and tipo == "FN"):
        assert not g.lesion_id.duplicated().any(), (red, tipo)
print("OK")

In [ ]:
# Ejemplo: primera figura de cada red.
from IPython.display import Image, display
for arch in ARCHS:
    f = casos[(casos.red == arch)].figura.iloc[0]
    display(Image(f"{SALIDA}/{f}"))

## Corrección de la prueba de cordura (solo si la ejecución se hizo con el commit `cb90485`)

Esa versión aleatorizaba una **copia** del modelo, y para EfficientNetB0 la copia no era idéntica al original (con la fracción 0 la correlación no daba 1). Ahora se aleatoriza el modelo **en sitio** y se restauran los pesos al terminar, con 20 imágenes en vez de 5. Esta celda repite solo la cordura y la registra en los runs `m3-gradcam-<arch>` ya creados (tag `cordura=v2`), sin tocar las figuras ni el CSV. Hace falta haber ejecutado antes las celdas de sesión, datos y MLflow, y haber hecho `git pull`.

In [ ]:
from src.casos import repetir_cordura
RUNS_M3 = {"resnet50": "83e76ccac1294e769440362b03d5526d",
           "densenet121": "04d66d41f00e48b39c3846c2cf6bc8f9",
           "efficientnetb0": "867249dd6fbd4a9c82b98ec588308015"}
filas = []
for arch, run_id in RUNS_M3.items():
    r = repetir_cordura(arch, cfg, run_id, n_cordura=20)
    for (metodo, f), rho in r["cordura"]["rho"].items():
        filas.append({"red": arch, "método": metodo, "fracción": f, "rho": round(rho, 3),
                      "vacíos": r["cordura"]["vacios"][(metodo, f)], "n": r["n"]})
tabla = pd.DataFrame(filas)
display(tabla.pivot_table(index=["red", "método"], columns="fracción", values="rho", dropna=False))
display(tabla.pivot_table(index=["red", "método"], columns="fracción", values="vacíos"))

## Siguiente paso (manual)

Anotar en `m3_casos.csv` (en Drive `TFM/m3/` y en el run `m3-gradcam-casos`): `zona_atencion` (lesión / borde / fuera: regla, esquina, pelo, burbuja, tinta / difusa), `criterios_abcde` (visibles) y, si hace falta, `comentario`.